In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# подготовка данных
import numpy as np
import pandas as pd
import re
import nltk
import string
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [8]:
df = pd.read_csv('/content/drive/MyDrive/reviews.csv', encoding='utf-8', sep='\t')
df.head()

,review,sentiment
0,качество плохое пошив ужасный (горловина напер...,negative
1,"Товар отдали другому человеку, я не получила п...",negative
2,"Ужасная синтетика! Тонкая, ничего общего с пре...",negative
3,"товар не пришел, продавец продлил защиту без м...",negative
4,"Кофточка голая синтетика, носить не возможно.",negative


In [9]:
df['cleaned_reviews'] = df['review'].str.lower() # приведение к нижнему регистру
df['cleaned_reviews'] = df['cleaned_reviews'].str.replace(r'[^\w\s]',' ', regex=True) # удаление пунктуации
df['cleaned_reviews'] = df['cleaned_reviews'].str.replace(r'\d+', '', regex=True) # удаление чисел
# удаление стоп-слов
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('russian'))
STOPWORDS -= {"не", "ни", "нет"}
def stopwords(text):
    return " ".join([word for word in str(text).split() if word not in STOPWORDS])
df["cleaned_reviews"] = df["cleaned_reviews"].apply(stopwords)
# удаление эмодзи
def emoji(string):
    emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', string)
#passing the emoji function to 'text_rare'
df['cleaned_reviews'] = df['cleaned_reviews'].apply(emoji)
# удалениe URL
def remove_urls(text):
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)
df['cleaned_reviews'] = df['cleaned_reviews'].apply(remove_urls)
df.head()

,review,sentiment,cleaned_reviews
0,качество плохое пошив ужасный (горловина напер...,negative,качество плохое пошив ужасный горловина напере...
1,"Товар отдали другому человеку, я не получила п...",negative,товар отдали другому человеку не получила посы...
2,"Ужасная синтетика! Тонкая, ничего общего с пре...",negative,ужасная синтетика тонкая общего представленной...
3,"товар не пришел, продавец продлил защиту без м...",negative,товар не пришел продавец продлил защиту моего ...
4,"Кофточка голая синтетика, носить не возможно.",negative,кофточка голая синтетика носить не возможно


In [10]:
# токенизация предобработанного текста
from nltk.tokenize import word_tokenize
nltk.download('punkt_tab')

def tokenize_text(text):
    return nltk.word_tokenize(text)

# сохраним токены (списки слов) в отдельной колонке tokenized_text
df['tokenized_text'] = df['cleaned_reviews'].apply(tokenize_text)
df.head()

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


,review,sentiment,cleaned_reviews,tokenized_text
0,качество плохое пошив ужасный (горловина напер...,negative,качество плохое пошив ужасный горловина напере...,"[качество, плохое, пошив, ужасный, горловина, ..."
1,"Товар отдали другому человеку, я не получила п...",negative,товар отдали другому человеку не получила посы...,"[товар, отдали, другому, человеку, не, получил..."
2,"Ужасная синтетика! Тонкая, ничего общего с пре...",negative,ужасная синтетика тонкая общего представленной...,"[ужасная, синтетика, тонкая, общего, представл..."
3,"товар не пришел, продавец продлил защиту без м...",negative,товар не пришел продавец продлил защиту моего ...,"[товар, не, пришел, продавец, продлил, защиту,..."
4,"Кофточка голая синтетика, носить не возможно.",negative,кофточка голая синтетика носить не возможно,"[кофточка, голая, синтетика, носить, не, возмо..."


In [11]:
# сохраним предобработанные данные в отдельный датафрейм
df_sample = df.groupby('sentiment', group_keys=False).sample(n=5000, random_state=42)
docs = df_sample['tokenized_text'].astype(str).tolist()
df_sample.head()

,review,sentiment,cleaned_reviews,tokenized_text
32308,"Есть зацепки,тонкий Ден.На рост 175 подошло.Од...",neautral,зацепки тонкий ден рост подошло одеть,"[зацепки, тонкий, ден, рост, подошло, одеть]"
52404,одноразовый. не рекомендую,neautral,одноразовый не рекомендую,"[одноразовый, не, рекомендую]"
53397,Платье нормальное.. Но были проблемы с доставк...,neautral,платье нормальное проблемы доставкой вместо об...,"[платье, нормальное, проблемы, доставкой, вмес..."
55058,Время доставки,neautral,время доставки,"[время, доставки]"
32664,Резинка плохо крепится на украшение.После нес...,neautral,резинка плохо крепится украшение нескольких пр...,"[резинка, плохо, крепится, украшение, нескольк..."


In [12]:
!pip install bertopic
from bertopic import BERTopic

topic_model = BERTopic(
    language="multilingual",
    # nr_topics=10,         # сделает тем меньше и крупнее
    # min_topic_size=300,
    calculate_probabilities=True,          # <-- ограничение (укрупнение) тем
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)

2026-02-28 11:15:03,161 - BERTopic - Embedding - Transforming documents to embeddings.


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

2026-02-28 11:15:36,657 - BERTopic - Embedding - Completed ✓
2026-02-28 11:15:36,659 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-28 11:16:11,805 - BERTopic - Dimensionality - Completed ✓
2026-02-28 11:16:11,806 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-28 11:16:41,076 - BERTopic - Cluster - Completed ✓
2026-02-28 11:16:41,085 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-28 11:16:41,395 - BERTopic - Representation - Completed ✓


In [13]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,7612,-1_не_очень_размер_это,"[не, очень, размер, это, продавец, ткань, каче...","[['заказ', 'пришел', 'очень', 'быстро', 'течен..."
1,0,340,0_цвет_цвета_белый_серый,"[цвет, цвета, белый, серый, синий, заказывала,...","[['размер', 'цвет', 'не', 'соответствует'], ['..."
2,1,281,1_хорошее_хорошие_хорошая_хороший,"[хорошее, хорошие, хорошая, хороший, качество,...","[['качество', 'хорошее', 'подошло', 'размер'],..."
3,2,253,2_фото_фотографии_соответствует_видно,"[фото, фотографии, соответствует, видно, факту...","[['фото', 'качество', 'хорошее'], ['ткань', 'н..."
4,3,246,3_ужасное_ужасная_ужасно_ужасный,"[ужасное, ужасная, ужасно, ужасный, ужасные, п...","[['ужасное', 'качество', 'швы', 'материал', 'о..."
...,...,...,...,...,...
177,176,10,176_исправила_задрать_ибанутый_порезы,"[исправила, задрать, ибанутый, порезы, уточнил...","[['доставка', 'быстрая', 'отправка', 'отправко..."
178,177,10,177_коммуникабельный_лепить__сюрпрайз_откащал,"[коммуникабельный, лепить_, сюрпрайз, откащал,...","[['продавец', 'место', 's', 'отправил', 'l', '..."
179,178,10,178_хлопок_жуткая_синтетика_синтетик,"[хлопок, жуткая, синтетика, синтетик, флисовой...","[['это', 'не', 'хлопок', 'синтетика', 'неприят..."
180,179,10,179_воротничок_грязный_трикотаж_качественно,"[воротничок, грязный, трикотаж, качественно, i...","[['качество', 'материала', 'хорошее', 'рисунок..."


In [14]:
# ключевые слова для темы 1 и их вес
topic_model.get_topic(1)

[('хорошее', np.float64(0.057139159637367196)),
 ('хорошие', np.float64(0.04869999919081548)),
 ('хорошая', np.float64(0.03515133892932456)),
 ('хороший', np.float64(0.02989667948793994)),
 ('качество', np.float64(0.021795456683177037)),
 ('хорошего', np.float64(0.021669071234298627)),
 ('качества', np.float64(0.0122311000257378)),
 ('футболка', np.float64(0.012090616655595845)),
 ('размер', np.float64(0.011946966166300766)),
 ('немного', np.float64(0.011165907854069212))]

In [15]:
# карта дистанций между темами
topic_model.visualize_topics()

In [16]:
# дендрограмма топ-50 тем
topic_model.visualize_hierarchy(top_n_topics=50)

In [17]:
# наиболее значимые слова в топ-5 темах
topic_model.visualize_barchart(top_n_topics=5)

In [18]:
# матрица сходств тем
topic_model.visualize_heatmap(n_clusters=20, width=1000, height=1000)


# **Результаты тематического моделирования**
Для обучения модели было выбрано 5000 отзывов каждого вида: positive, negative и neutral. BERTopic выделил 182 различных темы, однако большое количество документов (7743) не удалось классифицировать ни в одну из конкретных тем. Возможно, на это повлияло моё решение не избавляться от отрицательных частиц "не", "ни" и "нет" во время предобработки, так как они могут существенно поменять смысл написанного.

Тем не менее, модель хорошо справилась с разделением отзывов на темы, часто встречающиеся в отзывах пользователей: цвет (тема 0), соответствие фото (тема 1), размерная сетка (тема 2), возврат и доставка товара (тема 3) и т.д. Поскольку условия задания не предусматривали какие-либо ограничение на количетсво тем, было решено оставить все, в том числе микро-темы.


